In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.utils import plot_model

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
# Initialize random seeds for reproducibility

dictionary = pd.read_csv('FINANCE384_stock_month_data_dictionary.csv')
display(dictionary)

/Users/thismac/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,variable,description,role
0,date,End of month t (CRSP month-end trading date),identifier; not a predictor
1,permno,CRSP permanent security identifier,identifier; not a predictor
2,ticker,Exchange ticker at month t,identifier; not a predictor
3,comnam,Company name at month t,identifier; not a predictor
4,industry,Coarse SIC division label (Agriculture collaps...,"classification, known at t; permitted as a pre..."
5,ff49_code,Fama-French 49-industry code (Ships collapsed ...,"classification, known at t; permitted as a pre..."
6,ff49_name,Fama-French 49-industry name (Ships collapsed ...,"classification, known at t; permitted as a pre..."
7,gvkey,Compustat firm identifier (missing if no CCM l...,identifier; not a predictor
8,size,log market capitalization (USD) at end of mont...,"characteristic, contemporaneous with t"
9,bm,Book equity (6-month availability lag) / marke...,"characteristic, contemporaneous with t"


In [3]:
determined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, contemporaneous with t", "variable"].tolist()

predetermined_characteristics = dictionary.loc[
    dictionary["role"] == "characteristic, predetermined at t-1", "variable"].tolist()

determined_classifications = dictionary.loc[
    dictionary["role"] == "classification, known at t; permitted as a predictor if encoded", "variable"].tolist()

print(f"Rows: {len(dictionary):,}")
print(f"Determined characteristics: {determined_characteristics}")
print(f"Predetermined characteristics: {predetermined_characteristics}")
print(f"Determined classifications: {determined_classifications}")

FEATURES = determined_characteristics + predetermined_characteristics
print("Number of features:", len(FEATURES))

Rows: 27
Determined characteristics: ['size', 'bm', 'vol12', 'beta60', 'ivol60', 'turnover', 'dollar_volume', 'amihud_illiq', 'divyield', 'mkt_12m', 'mkt_vol_12m', 'down_market']
Predetermined characteristics: ['mom12_2', 'gross_profit', 'roe', 'asset_growth', 'leverage', 'accruals']
Determined classifications: ['industry', 'ff49_code', 'ff49_name']
Number of features: 18


In [4]:
panel = pd.read_csv('FINANCE384_assignmentA_development_panel.csv', parse_dates=['date'])

print(f"Rows           : {len(panel):,}")
print(f"Distinct stocks: {panel['permno'].nunique():,}")
print(f"Months         : {panel['date'].nunique():,} "
      f"({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
panel.head()


print(f"Duplicate (permno, date) rows: {panel.duplicated(['permno', 'date']).sum()}")

# Check 2: how many stocks are in each month, and how many months does a stock survive?
stocks_per_month = panel.groupby("date")["permno"].nunique()
months_per_stock = panel.groupby("permno")["date"].nunique()
print(f"Stocks per month : {stocks_per_month.min()} to {stocks_per_month.max()} "
      f"(mean {stocks_per_month.mean():.0f})")
print(f"Months per stock : {months_per_stock.min()} to {months_per_stock.max()} "
      f"(median {months_per_stock.median():.0f})")


Rows           : 198,298
Distinct stocks: 1,252
Months         : 396 (1990-01 to 2022-12)
Duplicate (permno, date) rows: 0
Stocks per month : 497 to 505 (mean 501)
Months per stock : 1 to 396 (median 122)


In [5]:
TARGET = "target_ret_excess_tp1"

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[["permno", "month", "ret_excess_t"]]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(next_month_return, on=["permno", "month"], how="left", validate="one_to_one")

final_month = (
    panel.groupby("permno")["month"]
    .transform("max")
    .eq(panel["month"])
)

no_target = panel[TARGET].isna()
# Diagnostics
print(f"Rows: {len(panel):,}")
print(f"Usable targets: {panel[TARGET].notna().sum():,}")
print(f"No month t+1 in data: {no_target.sum():,}")
print(f"Stock's last month: {(no_target & final_month).sum():,}")
print(f"Gap in coverage: {(no_target & ~final_month).sum():,}")

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())

print("\nFeature summary:")
display(
    panel[FEATURES + [TARGET]]
    .describe()
    .loc[["mean", "std", "min", "max"]]
)

panel[["permno", "month", "ret_excess_t", TARGET]].head()

Rows: 198,298
Usable targets: 197,016
No month t+1 in data: 1,282
Stock's last month: 1,252
Gap in coverage: 30

Missing values:
size                         0
bm                        4242
vol12                     5268
beta60                   17211
ivol60                   17211
turnover                     0
dollar_volume                0
amihud_illiq                 0
divyield                  5927
mkt_12m                      0
mkt_vol_12m                  0
down_market                  0
mom12_2                   6023
gross_profit               515
roe                       4259
asset_growth               737
leverage                   467
accruals                  6615
target_ret_excess_tp1     1282
dtype: int64

Feature summary:


,size,bm,vol12,beta60,ivol60,turnover,dollar_volume,amihud_illiq,divyield,mkt_12m,mkt_vol_12m,down_market,mom12_2,gross_profit,roe,asset_growth,leverage,accruals,target_ret_excess_tp1
mean,23.0467,0.5098,0.0847,1.0445,0.0725,0.1647,3.3957e+09,0.0003,0.0207,0.0891,0.0404,0.2297,0.1158,0.2994,0.1733,0.1099,0.2625,-0.0482,0.0083
std,1.3306,0.4449,0.0452,0.5579,0.0304,0.1518,6.9844e+09,0.0018,0.0196,0.1661,0.0164,0.4207,0.3274,0.2254,0.2518,0.2472,0.1713,0.0555,0.0990
min,17.3590,0.0082,0.0193,-0.3211,0.0257,0.0104,9.6556e+05,0.0000,0.0000,-0.4330,0.0108,0.0000,-0.9108,-0.1985,-0.8534,-0.4729,0.0000,-0.6748,-0.8703
max,27.5092,11.2483,0.5192,4.0973,0.2498,2.1366,2.3617e+11,0.1307,0.5481,0.6466,0.0894,1.0000,4.2891,1.1964,4.5006,3.0477,1.0184,0.1612,2.6266


,permno,month,ret_excess_t,target_ret_excess_tp1
0,10057,1990-01,-0.0540,-0.0179
1,10057,1990-02,-0.0179,-0.0558
2,10057,1990-03,-0.0558,-0.0485
3,10057,1990-04,-0.0485,-0.0205
4,10057,1990-05,-0.0205,-0.1035


In [6]:
TARGET_A = TARGET
TARGET = "y_risk"
panel[TARGET] = np.log1p(panel[TARGET_A] ** 2)
assert panel[TARGET].notna().equals(panel[TARGET_A].notna())
assert np.isfinite(panel.loc[panel[TARGET].notna(), TARGET]).all()


In [7]:
industry = 'ff49_name'

for feature in FEATURES:
    monthly_median = (
        panel.groupby('date')[feature]
        .transform('median'))
    
    panel[feature] = (
        panel[feature]
        .fillna(monthly_median))

panel = panel.loc[panel[TARGET].notna()].copy()

print("\nMissing values:")
print(panel[FEATURES + [TARGET]].isna().sum())


Missing values:
size             0
bm               0
vol12            0
beta60           0
ivol60           0
turnover         0
dollar_volume    0
amihud_illiq     0
divyield         0
mkt_12m          0
mkt_vol_12m      0
down_market      0
mom12_2          0
gross_profit     0
roe              0
asset_growth     0
leverage         0
accruals         0
y_risk           0
dtype: int64


In [8]:
train = panel.loc[panel['month'].between('1990-01', '2014-12')].copy()
valid = panel.loc[panel['month'].between('2015-01', '2018-12')].copy()
test = panel.loc[panel['month'].between('2019-01', '2022-11')].copy()
train_valid = pd.concat([train, valid]).copy()

split_summary = pd.DataFrame([
    {'sample': name, 'months': block['month'].nunique(), 'rows': len(block),
     'start': str(block['month'].min()), 'end': str(block['month'].max())}
    for name, block in [('Train', train), ('Validation', valid), ('Test', test)]
])
display(split_summary)


,sample,months,rows,start,end
0,Train,300,149334,1990-01,2014-12
1,Validation,48,24051,2015-01,2018-12
2,Test,47,23631,2019-01,2022-11


In [9]:
part_a_rows = {"Train": 149334, "Validation": 24051, "Test": 23631}
part_b_rows = {"Train": len(train), "Validation": len(valid), "Test": len(test)}
sample_check = pd.DataFrame({"Part A rows": part_a_rows, "Part B rows": part_b_rows})
sample_check["Match"] = sample_check["Part A rows"].eq(sample_check["Part B rows"])
display(sample_check)
assert sample_check["Match"].all()


,Part A rows,Part B rows,Match
Train,149334,149334,True
Validation,24051,24051,True
Test,23631,23631,True


In [10]:
def standardize(block, fitted_on):
    return (
        block[FEATURES]
        - fitted_on[FEATURES].mean()
    ) / fitted_on[FEATURES].std(ddof=0)


X_train = standardize(train, fitted_on=train)
X_valid = standardize(valid, fitted_on=train)
X_test = standardize(test, fitted_on=train)
X_train_valid = standardize(train_valid, fitted_on=train)

y_train, y_valid, y_test = train[TARGET], valid[TARGET], test[TARGET]
y_train_valid = train_valid[TARGET]

ols_benchmark = LinearRegression().fit(
    X_train_valid,
    y_train_valid
)

print("Number of coefficients:", len(ols_benchmark.coef_))
print("Intercept:", ols_benchmark.intercept_)

Number of coefficients: 18
Intercept: 0.009756719789081644


In [11]:
def neural_network(input):
    model = Sequential()
    model.add(
        Dense(
            16,
            input_shape=(input,),
            activation="relu"))
    model.add(
        Dense(
            1,
            activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mean_squared_error")
    model.summary()
    return model


## Train Neural Network

Runs a neural network experiment using a specified batch size, random seed, and maximum number of epochs. Early stopping monitors validation loss and restores the best-performing model weights. Training and validation MSE and RMSE are recorded, with the best validation result stored for model comparison.

In [12]:

def run_network(batch_size, SEED, EPOCHS):

    # Ensure every experiment begins from the same random seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(len(FEATURES))

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True
            )
        ],
        verbose=1,
        shuffle=False
    )

    # Create the error table
    error_table = pd.DataFrame({
        "Batch Size": batch_size,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"]
    )

    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"]
    )
    print(f"Batch size: {batch_size}, "
        f"Seed: {SEED}, "
        f"Maximum epochs: {EPOCHS}\n")
    display(error_table.set_index("Epoch"))

    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)



## Compare Batch Sizes

Trains the neural network using batch sizes of 32 and 16 while keeping the seed and maximum epochs fixed. The models are compared using their best validation RMSE.

In [13]:
# comparing batch sizes
model_results = []

run_network(32, 200, 100)   
run_network(16, 200, 100)
model_results = pd.DataFrame(model_results)
display(model_results.sort_values(by="Validation RMSE").reset_index(drop=True))

/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

...

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 252us/step - loss: 0.2306 - val_loss: 0.0013
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 248us/step - loss: 0.0015 - val_loss: 2.1393e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 250us/step - loss: 7.5154e-04 - val_loss: 1.8273e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 255us/step - loss: 6.6498e-04 - val_loss: 1.6651e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 257us/step - loss: 5.7697e-04 - val_loss: 1.6041e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 260us/step - loss: 5.5064e-04 - val_loss: 1.5415e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 260us/step - loss: 5.4013e-04 - val_loss: 1.5582e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 249us/step - loss: 5.3368e-04 - val_loss: 1.5817e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 248us/step - loss: 5.3074e-04 - val_loss: 1.5605e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 248us/step - loss: 5.2999e-04 - val_loss: 1.5440e-04
Epoch 11/100
9334/933

,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,
1,16,200,0.0582,0.0013,0.2412,0.0364
2,16,200,0.0015,0.0002,0.0384,0.0146
3,16,200,0.0010,0.0002,0.0309,0.0135
4,16,200,0.0008,0.0002,0.0289,0.0129
5,16,200,0.0008,0.0002,0.0280,0.0127
6,16,200,0.0008,0.0002,0.0276,0.0124
7,16,200,0.0008,0.0002,0.0275,0.0125
8,16,200,0.0008,0.0002,0.0274,0.0126
9,16,200,0.0008,0.0002,0.0274,0.0125


,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,16.0,200.0,6.0,0.0008,0.0002,0.0276,0.0124
1,32.0,200.0,18.0,0.0007,0.0002,0.0272,0.0124


## Neural Network Architecture Search

Defines a flexible neural network that allows the number of hidden layers, neurons, and learning rate to be varied. Each model uses the same batch size and random seed, with early stopping based on validation loss. Training and validation RMSE are recorded to compare the different model specifications.

In [14]:
EPOCHS = 100
BATCH_SIZE = 16
SEED = 200

def neural_network(input, hidden_layers, learning_rate):
    model = Sequential()

    # First hidden layer
    model.add(
        Dense(
            hidden_layers[0],
            input_shape=(input,),
            activation="relu"))

    # Any additional hidden layers
    for neurons in hidden_layers[1:]:
        model.add(
            Dense(
                neurons,
                activation="relu"))

    # One predicted return
    model.add(
        Dense(
            1,
            activation="linear"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mean_squared_error")
    return model


def run_network(hidden_layers, learning_rate):

    # Give every candidate the same starting seed
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)

    # Create a fresh network
    model = neural_network(
        len(FEATURES),
        hidden_layers,
        learning_rate)

    # Train the network
    history = model.fit(
        X_train.values,
        y_train.values,
        validation_data=(
            X_valid.values,
            y_valid.values
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True)
        ],
        verbose=1,
        shuffle=False)

    # Store the error from every completed epoch
    error_table = pd.DataFrame({
        "Hidden Layers": str(hidden_layers),
        "Learning Rate": learning_rate,
        "Batch Size": BATCH_SIZE,
        "Seed": SEED,
        "Epoch": range(
            1,
            len(history.history["loss"]) + 1
        ),
        "Training MSE": history.history["loss"],
        "Validation MSE": history.history["val_loss"]
    })
    error_table["Training RMSE"] = np.sqrt(
        error_table["Training MSE"])
    error_table["Validation RMSE"] = np.sqrt(
        error_table["Validation MSE"])
            
    # Display the complete table
    display(
        error_table
        .set_index("Epoch")
        .round(6))


    best_row = error_table.loc[
    error_table["Validation RMSE"].idxmin()]
    model_results.append(best_row)

    return model, history, error_table


In [15]:
model_results = []
run_network([32, 16, 8], 0.0015)
run_network([32, 16, 8], 0.001)
run_network([32, 16, 8], 0.0005)

run_network([16, 8], 0.0015)
run_network([16, 8], 0.001)
run_network([16, 8], 0.0005)

run_network([32], 0.001)
run_network([16], 0.001)


model_results = pd.DataFrame(model_results)
display(
    model_results
    .sort_values("Validation RMSE")
    .reset_index(drop=True)
)


Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 294us/step - loss: 0.0011 - val_loss: 1.6118e-04
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 281us/step - loss: 5.4255e-04 - val_loss: 1.5581e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 286us/step - loss: 5.3409e-04 - val_loss: 1.5295e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 279us/step - loss: 5.2316e-04 - val_loss: 1.5328e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 284us/step - loss: 5.2281e-04 - val_loss: 1.6168e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 284us/step - loss: 5.2031e-04 - val_loss: 1.5503e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 279us/step - loss: 5.2059e-04 - val_loss: 1.5843e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 282us/step - loss: 5.2290e-04 - val_loss: 1.5942e-04


,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,"[32, 16, 8]",0.0015,16,200,0.0009,0.0002,0.0297,0.0127
2,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0277,0.0125
3,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0275,0.0124
4,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0274,0.0124
5,"[32, 16, 8]",0.0015,16,200,0.0008,0.0002,0.0274,0.0127
6,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0273,0.0125
7,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0272,0.0126
8,"[32, 16, 8]",0.0015,16,200,0.0007,0.0002,0.0273,0.0126


Epoch 1/100


...

Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 272us/step - loss: 0.2306 - val_loss: 0.0013
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 2s 265us/step - loss: 0.0015 - val_loss: 2.1393e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 269us/step - loss: 7.5154e-04 - val_loss: 1.8273e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 268us/step - loss: 6.6498e-04 - val_loss: 1.6651e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 273us/step - loss: 5.7697e-04 - val_loss: 1.6041e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 272us/step - loss: 5.5064e-04 - val_loss: 1.5415e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 279us/step - loss: 5.4013e-04 - val_loss: 1.5582e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 294us/step - loss: 5.3368e-04 - val_loss: 1.5817e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 309us/step - loss: 5.3074e-04 - val_loss: 1.5605e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 306us/step - loss: 5.2999e-04 - val_loss: 1.5440e-04
Epoch 11/100
9334/9334 ━━━━━━━━━━

,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,[16],0.001,16,200,0.0582,0.0013,0.2412,0.0364
2,[16],0.001,16,200,0.0015,0.0002,0.0384,0.0146
3,[16],0.001,16,200,0.0010,0.0002,0.0309,0.0135
4,[16],0.001,16,200,0.0008,0.0002,0.0289,0.0129
5,[16],0.001,16,200,0.0008,0.0002,0.0280,0.0127
6,[16],0.001,16,200,0.0008,0.0002,0.0276,0.0124
7,[16],0.001,16,200,0.0008,0.0002,0.0275,0.0125
8,[16],0.001,16,200,0.0008,0.0002,0.0274,0.0126
9,[16],0.001,16,200,0.0008,0.0002,0.0274,0.0125


,Hidden Layers,Learning Rate,Batch Size,Seed,Epoch,Training MSE,Validation MSE,Training RMSE,Validation RMSE
0,[32],0.0010,16,200,13,0.0007,0.0002,0.0273,0.0123
1,"[32, 16, 8]",0.0015,16,200,3,0.0008,0.0002,0.0275,0.0124
2,"[32, 16, 8]",0.0005,16,200,4,0.0007,0.0002,0.0271,0.0124
3,[16],0.0010,16,200,6,0.0008,0.0002,0.0276,0.0124
4,"[16, 8]",0.0010,16,200,3,0.0008,0.0002,0.0278,0.0125
5,"[32, 16, 8]",0.0010,16,200,1,0.0009,0.0002,0.0304,0.0125
6,"[16, 8]",0.0005,16,200,5,0.0008,0.0002,0.0275,0.0125
7,"[16, 8]",0.0015,16,200,3,0.0008,0.0002,0.0278,0.0125


In [16]:
import ast
selected_hidden_layers = [32]
selected_learning_rate = 0.001
model_results = []

selected_model, selected_history, selected_error_table = run_network(
    selected_hidden_layers,
    selected_learning_rate
)

validation_predictions = selected_model.predict(
    X_valid.values,
    verbose=0
).flatten()


ols_predictions = ols_benchmark.predict(
    X_test.values
)

nn_predictions = selected_model.predict(
    X_test.values,
    verbose=0
).flatten()

valid_results = valid[
    ["date", "month", "permno", TARGET]
].copy()

test_results = test[
    ["date", "month", "permno", TARGET]
].copy()



valid_results["NN prediction"] = validation_predictions
test_results["OLS prediction"] = ols_predictions
test_results["NN prediction"] = nn_predictions



Epoch 1/100


/Users/thismac/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 320us/step - loss: 0.4397 - val_loss: 0.0024
Epoch 2/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 291us/step - loss: 0.0037 - val_loss: 8.2031e-04
Epoch 3/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 321us/step - loss: 0.0016 - val_loss: 2.2478e-04
Epoch 4/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 328us/step - loss: 0.0017 - val_loss: 1.8262e-04
Epoch 5/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 302us/step - loss: 0.0011 - val_loss: 1.6662e-04
Epoch 6/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 300us/step - loss: 8.2482e-04 - val_loss: 1.6206e-04
Epoch 7/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 303us/step - loss: 6.2110e-04 - val_loss: 1.5340e-04
Epoch 8/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 301us/step - loss: 5.4261e-04 - val_loss: 1.5290e-04
Epoch 9/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 298us/step - loss: 5.3255e-04 - val_loss: 1.5352e-04
Epoch 10/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3s 282us/step - loss: 5.2897e-04 - val_loss: 1.5402e-04
Epoch 11/100
9334/9334 ━━━━━━━━━━━━━━━━━━━━ 3

,Hidden Layers,Learning Rate,Batch Size,Seed,Training MSE,Validation MSE,Training RMSE,Validation RMSE
Epoch,,,,,,,,
1,[32],0.001,16,200,0.0936,0.0024,0.3059,0.0487
2,[32],0.001,16,200,0.0030,0.0008,0.0549,0.0286
3,[32],0.001,16,200,0.0015,0.0002,0.0381,0.0150
4,[32],0.001,16,200,0.0013,0.0002,0.0356,0.0135
5,[32],0.001,16,200,0.0010,0.0002,0.0322,0.0129
6,[32],0.001,16,200,0.0009,0.0002,0.0293,0.0127
7,[32],0.001,16,200,0.0008,0.0002,0.0283,0.0124
8,[32],0.001,16,200,0.0008,0.0002,0.0276,0.0124
9,[32],0.001,16,200,0.0008,0.0002,0.0274,0.0124


/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/thismac/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


*** RMSE

In [17]:

valid_rmse = np.sqrt(
    mean_squared_error(
        valid_results[TARGET],
        valid_results["NN prediction"]
    )
)



nn_test_rmse = np.sqrt(
    mean_squared_error(
        test_results[TARGET],
        test_results["NN prediction"]
    )
)

ols_test_rmse = np.sqrt(
    mean_squared_error(
        test_results[TARGET],
        test_results["OLS prediction"]
    )
)



In [18]:
def mean_monthly_spearman(df, actual_col, prediction_col):

    monthly_correlations = (
        df.groupby("month")
        .apply(
            lambda x: spearmanr(
                x[actual_col],
                x[prediction_col]
            ).statistic
        )
    )

    return monthly_correlations.mean()

valid_spearman = mean_monthly_spearman(
    valid_results,
    TARGET,
    "NN prediction"
)

nn_test_spearman = mean_monthly_spearman(
    test_results,
    TARGET,
    "NN prediction"
)

ols_test_spearman = mean_monthly_spearman(
    test_results,
    TARGET,
    "OLS prediction"
)

valid_months = valid_results["month"].nunique()
valid_rows = len(valid_results)


test_months = test_results["month"].nunique()
test_rows = len(test_results)

results_summary = pd.DataFrame({
    "Period": [
        "Validation",
        "Test",
        "Test"
    ],
    "Model": [
        "NN",
        "NN",
        "OLS"
    ],
    "Months": [
        valid_months,
        test_months,
        test_months
    ],
    "Stock-month rows": [
        valid_rows,
        test_rows,
        test_rows
    ],
    "Pooled RMSE": [
        valid_rmse,
        nn_test_rmse,
        ols_test_rmse
    ],
    "Mean Monthly Spearman": [
        valid_spearman,
        nn_test_spearman,
        ols_test_spearman
    ]
})

display(
    results_summary.round(6)
)

/var/folders/6m/txxx78_s0kl86355p_vdt9kh0000gn/T/ipykernel_13973/3364627460.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("month")
/var/folders/6m/txxx78_s0kl86355p_vdt9kh0000gn/T/ipykernel_13973/3364627460.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("month")
/var/folders/6m/txxx78_s0kl86355p_vdt9kh0000gn/T/ipykernel_13973/3364627460.py:4: FutureWarning: DataFrameGroupBy.apply oper

,Period,Model,Months,Stock-month rows,Pooled RMSE,Mean Monthly Spearman
0,Validation,NN,48,24051,0.0123,0.2182
1,Test,NN,47,23631,0.0244,0.2112
2,Test,OLS,47,23631,0.0240,0.2093
